# **Trích pose PHOENIX-2014T**

**Setup**: Settings → Accelerator: **None (CPU only)**. Internet: ON.

**Vì sao CPU-only**: MediaPipe Holistic chạy trên CPU, không dùng GPU.

`KAGGLE_NOTEBOOK_EXTRACT.ipynb` (train). Đây là bước 0, chỉ cần chạy 1 LẦN DUY NHẤT cho toàn bộ corpus, kết quả dùng lại mãi mãi cho mọi subset/mọi thí nghiệm về sau.

**Dataset cần add** (Add Input): `phoenix-2014t` (có thư mục
`features/fullFrame-210x260px/<split>/<name>/images0001.png ...` - KHÔNG có file video).
- image - `https://www.kaggle.com/datasets/thesisacc2/rwth-phoenix-2014-t/data`
- annotation - `https://www.kaggle.com/datasets/thesisacc2/phoenix-2014t-annotations/data`

## **Cell 1 - Cài dependencies**

In [ ]:
!pip uninstall -y -q tensorflow tensorflow-cpu tf-keras keras mediapipe mediapipe-nightly
!pip install -q "mediapipe==0.10.14" opencv-python-headless tqdm

## **Cell 2 - Lấy code**
Giống notebook train: clone GitHub hoặc copy từ Kaggle Dataset `slt-rl-code`.

In [ ]:
!rm -rf /kaggle/working/Sign-Language-REL_code

In [ ]:
import os
os.chdir("/kaggle/working")
!git clone https://github.com/linhxm/Sign-Language-REL_code.git
# !cp -r /kaggle/input/slt-rl-code/slt_rl /kaggle/working/
os.chdir("/kaggle/working/Sign-Language-REL_code")

## **Cell 3 - Smoke-test đường dẫn (BẮT BUỘC trước khi chạy full)**

10-20h là quá đắt để phát hiện sai đường dẫn ở cuối. Kỳ vọng in ra `Tìm thấy n thư mục ảnh ...`
rồi `--limit 5: chỉ xử lý 5 sequence đầu`.

In [ ]:
# !python data/extract_poses.py \
#     --input_dir /kaggle/input/datasets/thesisacc2/rwth-phoenix-2014-t \
#     --out_dir /kaggle/working/poses_test --limit 5

## **Cell 3b - Kiểm tra 1 file .npz vừa tạo**
Kỳ vọng shape `(T, 183)`, dtype `float32`. Nếu `tỉ lệ phần tử = 0` gần bằng 1.0 nghĩa là MediaPipe không detect được gì (path ảnh sai hoặc ảnh rỗng) - DỪNG LẠI kiểm tra trước khi chạy full.

In [ ]:
# import numpy as np, glob, os
# f = sorted(glob.glob("/kaggle/working/poses_test/*.npz"))[0]
# a = np.load(f)["pose"]
# print(os.path.basename(f), a.shape, a.dtype)
# print("tỉ lệ phần tử = 0:", (a == 0).mean())

## **Cell 4 - (resumable - script tự bỏ qua file `.npz` đã có)**

`--workers 0` = tự chọn `cpu_count()-1`. Script ghi từng `.npz` qua file `.tmp` rồi đổi tên, nên
bị kill giữa chừng (hết giờ session) KHÔNG để lại file hỏng - chạy lại đúng lệnh này ở version
sau là tiếp tục đúng chỗ dừng, không mất phần đã xong.

In [ ]:
!python data/extract_poses.py \
    --input_dir /kaggle/input/datasets/thesisacc2/rwth-phoenix-2014-t \
    --out_dir /kaggle/working/poses --workers 0 --num_shards 3 --shard 0

## **Cell 5 - Nếu cần RESUME ở session/version MỚI (hết giờ giữa chừng)**

`/kaggle/working` reset mỗi session mới - không tự động giữ `.npz` đã extract ở version trước.
Cách resume:
1. Sau khi version trước dừng (hết giờ), nó vẫn có 1 **Output** (`/kaggle/working/poses` của lần
   chạy đó) - vào tab Output của chính notebook này, bấm **"New Dataset"** để đóng gói phần đã
   extract thành 1 Kaggle Dataset tạm (vd `phoenix-poses-partial`).
2. Add Dataset đó làm Input cho version MỚI của notebook này.
3. Copy các `.npz` đã có vào lại `out_dir` TRƯỚC khi chạy lại Cell 4 (để script nhận ra và skip):
   `!mkdir -p /kaggle/working/poses && cp /kaggle/input/phoenix-poses-partial/*.npz /kaggle/working/poses/`
4. Chạy lại Cell 4 - chỉ những sequence CHƯA có `.npz` mới được xử lý tiếp.

**Chạy song song nhiều notebook** (nếu muốn rút ngắn thời gian tổng): dùng `--shard i --num_shards N`
trên N notebook khác nhau (vd N=2: mỗi cái `--num_shards 2 --shard 0` / `--shard 1`), rồi gộp
output của cả 2 lại trước khi đóng gói Dataset cuối.

## **Cell 6 - Sau khi xong: tạo Kaggle Dataset `phoenix-poses`**

Script tự in `missing_counts` (3 con số body/lhand/rhand - bằng chứng định lượng cho vấn đề chất
lượng pose, GHI LẠI 3 con số này cho báo cáo).

Sau khi Cell 4 chạy xong TOÀN BỘ (không còn sequence nào chưa xử lý): vào tab **Output** của
notebook này → **"New Dataset"** → đặt tên `phoenix-poses` → Add Input đúng dataset này vào
`KAGGLE_NOTEBOOK.ipynb` (notebook train) và trỏ `cfg.data.pose_cache_dir` (đã mặc định đúng
`/kaggle/input/phoenix-poses` trong `configs/config.py`).

In [ ]:
import os
n = len(os.listdir("/kaggle/working/poses"))
print(f"Tổng số file .npz đã có: {n} (kỳ vọng 8257 nếu PHOENIX-2014T full)")